In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from ydata_profiling import ProfileReport
import pickle

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 50

data_folder = "../data/"
images_folder = "../data/images/"
pipelines_folder = "../pipelines/"
df_total = pd.read_csv(os.path.join(data_folder, 'items_phase_1.csv'))
df_train = pd.read_csv(os.path.join(data_folder, 'items_train.csv'))
df_task_1 = pd.read_csv(os.path.join(data_folder, 'task_1.csv'))

# Notebook `items_phase_1.csv`

In [20]:
# profile = ProfileReport(df_total, title="items_phase_1", explorative=True)
# profile.to_notebook_iframe()

In [21]:
df_total.sample(4)

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo
87391,267806,36.00,231,['1'],NaN,Чехли adidas,adidas Чехли adilette Shower GZ3775 Бял,bg
70100,349732,16.24,6446,['11'],NaN,Celio Šortky Topiquebm - Muži,Monochromatické pletené šortky. Pevný pás s pú...,sk
8969,1264219,51.95,"6455,1443396",['1'],NaN,Sneakersy Dorko,Sneakersy Dorko Ultraforce DS24F08W Béžová,sk
10858,486863,24250.00,231,['11'],NaN,Sportcipők Aldo,Aldo Sportcipők Finespec 13451152 Fehér,hu


## Key takeaway
- missingy se musi resit u:
    - description (3%)
    - brandEditionTagId (99.8%) - to je asi target?
    - colorTagIdsString (3.1%)
---
# Notebook `task_1.csv`
- kazdy radek je jedna skupina se sloupci item - item4(to jsou id do items_phase_1.csv - itemId)
- Kazdy 

In [22]:
# profile = ProfileReport(df_task_1, title="task_1", explorative=True)
# profile.to_notebook_iframe()

In [23]:
df_task_1.head()

,item1,item2,item3,item4,item5
0,130622,292253,463442,483968,1253745
1,82627,388496,553738,638400,884327
2,46130,333489,644448,848154,1178149
3,150796,248537,742067,1206230,1280786
4,76610,196894,345145,620255,932761


---
# Dataset `items_train.csv`

In [24]:
print("Total records:", len(df_train))
print("Total null values:\n", df_train.isnull().sum())

Total records: 928234
Total null values:
 itemId                    0
price                     0
colorTagIdsString     27834
departmentIds             0
brandEditionTagId    925518
title                     0
description           35473
geo                       0
label                     0
dtype: int64


In [25]:
df_train.sample(4)

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo,label
595355,622266,54.00,232,['1'],NaN,Чехли Tommy Hilfiger Hilfiger Poolside With We...,NaN,bg,47126
319334,625513,64.90,778,['1'],NaN,TOMMY HILFIGER Pas pesek / marine,"Dizajn: Prešit obod/rob, Raven rob; Način zape...",si,46546
762441,1232085,171.00,NaN,['11'],NaN,Сникърси Timberland Winsor Trail Mid Lc TB0A41...,NaN,bg,25676
45968,850559,154.99,783,['1'],NaN,Сандали Gino Rossi,Gino Rossi Сандали 1915-766.925 Кафяв,bg,2494


In [26]:
# profile = ProfileReport(df_train, title="task_1", explorative=True)
# profile.to_notebook_iframe()


---
# Vycisteni dat 
- prevod na spolecnou menu 
- normalizace meny v danem geo uzemi
- doplneni null hodnot
- null ve sloupci `colorTagIdString` muzu nahradit 0 
- `colorTagIdString` a `departmentIds` je potreba roztrhnout - obsahuji vice hodnot oddelenych carkou

## Tvorba preprocessing pipeline

In [27]:
df_total[df_total["brandEditionTagId"] == 0]

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo


In [28]:
df_train.columns

Index(['itemId', 'price', 'colorTagIdsString', 'departmentIds',
       'brandEditionTagId', 'title', 'description', 'geo', 'label'],
      dtype='object')

In [29]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from PriceGeoTransformer import PriceGeoTransformer
from DepartmentIdsTransformer import DepartmentIdsCleaner
import numpy as np

imput_zero_cols = ['colorTagIdsString', "brandEditionTagId"]
input_zero_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
])

input_unknown_cols = ["description"]
input_unknown_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value="Unknown")),
])

categorical_features = ['geo',"price"]
categorical_transformer = Pipeline(steps=[
    ('PriceGeoTransformer', PriceGeoTransformer())
])


# prevadi na stejny format jako jsou barvy - cisla oddelena carkou 
department_features = ['departmentIds']
department_transformer = Pipeline(steps=[
    ('DepartmentIdsCleaner', DepartmentIdsCleaner())
])

# Combine preprocessing for numeric and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('zero', input_zero_transformer, imput_zero_cols),
        ('unknown', input_unknown_transformer, input_unknown_cols),
        ('geo', categorical_transformer, categorical_features),
        ('department', department_transformer, department_features)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

preprocessor.set_output(transform="pandas")



pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

df_transformed = pipeline.fit_transform(df_train)

In [30]:
pickle.dump(pipeline, open(os.path.join(pipelines_folder, 'preprocessing_pipeline.pkl'), 'wb'))

## Příprava pro PyTorch dataset
- PyTorch bude použit na vytvoření embeddingů sloupců s více kategorijema a na kategorické sloupce
- PyTorch umí totiž lépe tvořit řídké matice

In [ ]:
unique_geos = df_transformed['geo'].dropna().unique().tolist()
geo_to_idx = {geo: idx for idx, geo in enumerate(unique_geos)}

def build_id_mapping(series):
    all_ids = series.dropna().astype(str).str.split(',').explode().str.strip()
    all_ids = all_ids[all_ids != '']
    
    unique_ids = sorted(all_ids.unique())
    
    id_to_idx = {raw_id: idx for idx, raw_id in enumerate(unique_ids)}
    
    return id_to_idx

color_to_idx = build_id_mapping(df_transformed['colorTagIdsString'])
dept_to_idx = build_id_mapping(df_transformed['departmentIds'])

print(f"Total Unique Colors: {len(color_to_idx)}")
print(f"Total Unique Depts: {len(dept_to_idx)}")

MAX_COLOR_ID = len(color_to_idx) - 1
MAX_DEPT_ID = len(dept_to_idx) - 1

print(f"Max Color ID: {MAX_COLOR_ID}, Max Dept ID: {MAX_DEPT_ID}")

Total Unique Colors: 96
Total Unique Depts: 3
Max Color ID: 95, Max Dept ID: 2


## V tomto stavu:
- Nejsou null hodnoty ve sloupcích:
    - colorTagIdString
    - description
    -  brandEditionTagId
- Barvy a deptId maji stejny format - cisla oddelena carkou
- chybi udelat OneHot encodingy kategorickejch - to udelame v PyTorchi